In [1]:
%run ../pattern_recognition/input/Format.ipynb
import ROOT as root
from array import array
root.gErrorIgnoreLevel = root.kFatal
%jsroot on
import numpy as np

/home/yoren/.local/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Welcome to JupyROOT 6.30/06


Error in <TUnixSystem::FindDynamicLibrary>: input/logo/PHENIXTools/lib/libLogoPainter.so does not exist in /home/yoren/bnl/ROOT/install/lib:.:/home/yoren/bnl/ROOT/install/lib:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/lib/x86_64-linux-gnu/tls/haswell/x86_64:/lib/x86_64-linux-gnu/tls/haswell:/lib/x86_64-linux-gnu/tls/x86_64:/lib/x86_64-linux-gnu/tls:/lib/x86_64-linux-gnu/haswell/x86_64:/lib/x86_64-linux-gnu/haswell:/lib/x86_64-linux-gnu/x86_64:/lib/x86_64-linux-gnu:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/usr/lib/x86_64-linux-gnu/tls/haswell/x86_64:/usr/lib/x86_64-linux-gnu/tls/haswell:/usr/lib/x86_64-linux-gnu/tls/x86_64:/usr/lib/x86_64-linux-gnu/tls:/usr/lib/x86_64-linux-gnu/haswell/x86_64:/usr/lib/x86_64-linux-gnu/haswell:/usr/lib/x86_64-linux-gnu/x86_64:/usr/lib/x86_64-linux-gnu:/lib/glibc-hwcaps/x86-64-v3:/lib/glibc-hwcaps/x86-64-v2:/lib/tls/haswell/x86_64:/lib/tls/haswell:/lib

In [2]:
file_path="input/"
file_names=["75570_TpcNoEdges_pp2025Alignment_Phi_Layer.root"]

In [3]:
hists_names=["phiresbox_phi_layer_0","phiresbox_phi_layer_1","phiresbox_phi_layer_AllSectors_0","phiresbox_phi_layer_AllSectors_1"]

In [4]:
input_file = root.TFile(file_path+file_names[0])
hists = []
for hist_name in hists_names:
    hist = input_file.Get(hist_name)
    hists.append(hist)
    hists[-1].SetDirectory(root.nullptr) 
input_file.Close()

In [5]:
c1 = root.TCanvas("c1", "c1", 1200, 1200)
c1.Divide(2,2)
for i in range(4):
    c1.cd(i+1)
    hists[i].Draw("colz")
c1.Draw()

In [6]:
substracted_hists = []
for i in range(2):
    substracted_hist = hists[i].Rebin(1,hists[i].GetName() + "_substracted")
    substracted_hists.append(substracted_hist)
    substracted_hists[-1].SetDirectory(root.nullptr)

In [16]:
half_sec_width = np.pi/6

In [17]:
for i in range(2):
    for ibinX in range(1, substracted_hists[i].GetNbinsX()+1):
        for ibinY in range(1, substracted_hists[i].GetNbinsY()+1):
            bin_center = substracted_hists[i].GetYaxis().GetBinCenter(ibinY)
            if bin_center > half_sec_width:
                bin_center -= 2*half_sec_width
            elif bin_center < -half_sec_width:
                bin_center += 2*half_sec_width
            bin_content = substracted_hists[i].GetBinContent(ibinX, ibinY)
            new_bin_y = hists[i+2].GetYaxis().FindBin(bin_center)
            bin_content_to_substract = hists[i+2].GetBinContent(ibinX, new_bin_y)
            new_bin_content = bin_content - bin_content_to_substract
            substracted_hists[i].SetBinContent(ibinX, ibinY, new_bin_content)


In [23]:
c2 = root.TCanvas("c2", "c2", 1200, 600)
c2.Divide(2,1)
for i in range(2):
    c2.cd(i+1)
    substracted_hists[i].Draw("colz")
    substracted_hists[i].GetZaxis().SetRangeUser(-0.01, 0.01)
    #root.gPad.SetLogz()
c2.Draw()